<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!git clone https://github.com/uomna/flyrank-ml.git
%cd flyrank-ml
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 147 (delta 54), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.86 MiB | 11.98 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyrank-ml/flyrank-ml


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
list(df.columns)

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — Staleness vs CTR: MIXED
Freshness tier does not show a consistent direction with CTR. The 0-30d
tier has moderate CTR (0.61%), but 31-90d and 91-180d tiers are lower
(0.12% and 0.24%). The 181+ tier shows the highest CTR (3.69%), but this
group is very small (n=174) compared to the 0-30d tier (n=20,480) — likely
survivorship bias (only strong, well-performing old pages remain live),
not evidence that staleness improves performance.

In [11]:
signal1_bucket = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()
signal1_bucket

,freshness_tier,n,avg_ctr
0,0-30,20480,0.609021
1,181+,174,3.693276
2,31-90,175,0.117543
3,91-180,9171,0.238367


Signal 2 — Position vs CTR: CONFIRMED
CTR decreases consistently and monotonically as position gets worse.
Top 3 positions average 2.71% CTR (n=1,141), dropping to 0.65% at
positions 4-10 (n=11,842), 0.32% at 11-20 (n=7,273), and 0.21% at 21+
(n=8,524). This is a clean, monotonic relationship — unlike Signal 1,
there is no reversal or small-sample noise here. This confirms that
position alone does not guarantee clicks; pages ranking well but still
getting low CTR relative to their position tier are strong candidates
for review (e.g., weak title/meta description).

In [12]:
import pandas as pd

df_valid = df[df["avg_position"] > 0].copy()
df_valid["position_bucket"] = pd.cut(
    df_valid["avg_position"],
    bins=[0, 3, 10, 20, 100],
    labels=["1-3", "4-10", "11-20", "21+"]
)

signal2_bucket = df_valid.groupby("position_bucket", observed=True).agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()
signal2_bucket

,position_bucket,n,avg_ctr
0,1-3,1141,2.714303
1,4-10,11842,0.651045
2,11-20,7273,0.323443
3,21+,8524,0.211705


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule: A page is worth reviewing if it ranks well in Google (good average
position) but its click-through rate is below what pages at that same
position level normally get, AND it gets enough impressions to matter
(500+ in 90 days). The score is the size of that CTR gap multiplied by
how many people actually see the page — so pages with both a big gap
and a big audience rank highest.

Reason code: ctr_below_position_tier
Action: review_title_and_meta (the page ranks fine; the click-driver —
title/meta description — is likely the problem, not the content itself).

In [13]:
# 1) Compute the average CTR expected for each position tier (from Signal 2 work)
tier_avg_ctr = df_valid.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df_valid["tier_avg_ctr"] = tier_avg_ctr

# 2) Gap = how far below its tier's average CTR this page is
df_valid["ctr_gap"] = df_valid["tier_avg_ctr"] - df_valid["ctr"]

# 3) A page only counts if it's actually visible (readable condition, no fitted weights)
visible = (df_valid["impressions_90d"] >= 500).astype(int)
underperforming = (df_valid["ctr_gap"] > 0).astype(int)

# 4) The transparent score: gap size * visibility * whether it's underperforming
df_valid["score"] = underperforming * visible * df_valid["ctr_gap"] * df_valid["impressions_90d"]

# 5) Reason code
df_valid["reason_code"] = "ctr_below_position_tier"

# 6) Action label
df_valid["action"] = "review_title_and_meta"

# 7) Rank and save
ranked = df_valid.sort_values("score", ascending=False)
ranked_out = ranked[["content_id", "position_bucket", "avg_position", "ctr",
                      "tier_avg_ctr", "ctr_gap", "impressions_90d",
                      "score", "reason_code", "action"]]

import os
os.makedirs("work/outputs", exist_ok=True)
ranked_out.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked_out.head(10)

,content_id,position_bucket,avg_position,ctr,tier_avg_ctr,ctr_gap,impressions_90d,score,reason_code,action
26844,content_8c19996aa890,1-3,2.5,0.15,2.714303,2.564303,509252,1.305877e+06,ctr_below_position_tier,review_title_and_meta
21819,content_4c36c775b818,1-3,2.3,0.41,2.714303,2.304303,463103,1.067130e+06,ctr_below_position_tier,review_title_and_meta
7678,content_8451fc6f034d,1-3,2.3,0.03,2.714303,2.684303,272144,7.305170e+05,ctr_below_position_tier,review_title_and_meta
14090,content_44e481c8f55b,1-3,1.4,0.65,2.714303,2.064303,312694,6.454952e+05,ctr_below_position_tier,review_title_and_meta
21565,content_9532f197bbc8,1-3,2.0,0.87,2.714303,1.844303,309192,5.702438e+05,ctr_below_position_tier,review_title_and_meta
16736,content_e12868d1f396,1-3,2.9,0.07,2.714303,2.644303,149712,3.958839e+05,ctr_below_position_tier,review_title_and_meta
3331,content_4a6607efcb46,1-3,2.2,0.01,2.714303,2.704303,128068,3.463347e+05,ctr_below_position_tier,review_title_and_meta
3295,content_4fc39a2b8cf0,1-3,2.6,0.69,2.714303,2.024303,160959,3.258298e+05,ctr_below_position_tier,review_title_and_meta
2346,content_11900bd7941a,1-3,2.8,0.41,2.714303,2.304303,123561,2.847220e+05,ctr_below_position_tier,review_title_and_meta
18803,content_03d2673b2553,1-3,1.9,0.83,2.714303,1.884303,143314,2.700470e+05,ctr_below_position_tier,review_title_and_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_8c19996aa890 — Action: review_title_and_meta. Ranks #2.5 but
   CTR is 0.15% vs expected 2.71% — a huge gap with 509K impressions, the
   biggest opportunity in the list. What could make this wrong: the
   query may be informational and the snippet already answers it, so
   people don't need to click.

2. content_4c36c775b818 — Action: review_title_and_meta. Ranks #2.3, CTR
   0.41% vs 2.71% expected, 463K impressions. What could make this wrong:
   a competitor's result above/below it may look more compelling — a
   position-only comparison can't see the surrounding SERP.

3. content_8451fc6f034d — Action: review_title_and_meta. Ranks #2.3, CTR
   is nearly zero (0.03%) vs 2.71% expected. What could make this wrong:
   this could be a tracking/measurement issue (e.g. broken click tracking)
   rather than an actual title problem — worth a sanity check before acting.

4. content_44e481c8f55b — Action: review_title_and_meta. Ranks #1.4 (near
   top spot) yet CTR is only 0.65%. What could make this wrong: top-1
   pages sometimes lose clicks to featured snippets/People Also Ask boxes
   above them, which position data alone doesn't capture.

5. content_9532f197bbc8 — Action: review_title_and_meta. Ranks #2.0, CTR
   0.87% vs 2.71% expected, still a real gap. What could make this wrong:
   the gap here is smaller than others in the list — this may be closer
   to normal variation than a true problem page.

6. content_e12868d1f396 — Action: review_title_and_meta. Ranks #2.9, CTR
   0.07%, 149K impressions. What could make this wrong: lower impressions
   than the top entries, so the same fix here matters less in absolute terms
   even though the percentage gap is large.

7. content_4a6607efcb46 — Action: review_title_and_meta. Ranks #2.2, CTR
   almost zero (0.01%). What could make this wrong: same as #3 — an
   extremely low CTR like this is a candidate for a data/tracking check,
   not just a content fix.

8. content_4fc39a2b8cf0 — Action: review_title_and_meta. Ranks #2.6, CTR
   0.69% vs 2.71% expected. What could make this wrong: this client's
   audience or intent may just click less on average (e.g. branded
   navigational searches), which the position-tier average doesn't adjust for.

9. content_11900bd7941a — Action: review_title_and_meta. Ranks #2.8, CTR
   0.41%, 123K impressions — the lowest impressions in the top 10. What
   could make this wrong: lower reach means fixing this page has the
   smallest upside of the group despite making the ranked list.

10. content_03d2673b2553 — Action: review_title_and_meta. Ranks #1.9, CTR
    0.83% vs 2.71% expected. What could make this wrong: like #5, this is
    one of the smaller gaps in the top 10 — worth double-checking it's a
    real problem and not normal day-to-day fluctuation.

In [14]:
top10 = ranked_out.head(10)
for i, row in top10.iterrows():
    print(f"{row['content_id']}: position={row['avg_position']}, "
          f"ctr={row['ctr']}%, expected={row['tier_avg_ctr']:.2f}%, "
          f"impressions={row['impressions_90d']}")

content_8c19996aa890: position=2.5, ctr=0.15%, expected=2.71%, impressions=509252
content_4c36c775b818: position=2.3, ctr=0.41%, expected=2.71%, impressions=463103
content_8451fc6f034d: position=2.3, ctr=0.03%, expected=2.71%, impressions=272144
content_44e481c8f55b: position=1.4, ctr=0.65%, expected=2.71%, impressions=312694
content_9532f197bbc8: position=2.0, ctr=0.87%, expected=2.71%, impressions=309192
content_e12868d1f396: position=2.9, ctr=0.07%, expected=2.71%, impressions=149712
content_4a6607efcb46: position=2.2, ctr=0.01%, expected=2.71%, impressions=128068
content_4fc39a2b8cf0: position=2.6, ctr=0.69%, expected=2.71%, impressions=160959
content_11900bd7941a: position=2.8, ctr=0.41%, expected=2.71%, impressions=123561
content_03d2673b2553: position=1.9, ctr=0.83%, expected=2.71%, impressions=143314


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:
content_8451fc6f034d and content_4a6607efcb46 both have near-zero CTR
(0.03% and 0.01%) despite ranking in the top 3 positions with 272K and
128K impressions. A CTR this close to zero at that position is unusual
enough to suspect a data or tracking issue (e.g. broken click tracking,
a redirect, or a page that no longer matches the ranking query) rather
than simply "the title is bad." These should be manually verified in
Search Console before assuming a title/meta rewrite will fix them —
the baseline score cannot tell the difference between a content problem
and a measurement problem.

Leakage check:
Confirmed programmatically — trend_direction and trend_pct were never
used as inputs to the score. Only avg_position, ctr, position_bucket,
and impressions_90d were used.

In [15]:
# Leakage check — confirm forbidden columns were never used as inputs
forbidden = ["trend_direction", "trend_pct"]
score_inputs = ["avg_position", "ctr", "position_bucket", "impressions_90d"]

leak_found = [col for col in forbidden if col in score_inputs]
print("Leakage check:", "PASSED - no forbidden columns used" if not leak_found else f"FAILED: {leak_found}")

# Weak picks — flag rows where the near-zero CTR looks like a tracking issue, not a real gap
top10 = ranked_out.head(10)
suspect = top10[top10["ctr"] < 0.05]
print("\nSuspect rows (CTR near-zero, possible tracking issue):")
print(suspect[["content_id", "ctr", "impressions_90d"]])

Leakage check: PASSED - no forbidden columns used

Suspect rows (CTR near-zero, possible tracking issue):
                content_id   ctr  impressions_90d
7678  content_8451fc6f034d  0.03           272144
3331  content_4a6607efcb46  0.01           128068


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.